# Assignment 3: CIFAR-10 Classification with YOLO


## Objective

Use the CIFAR-10 dataset with a YOLO classification model to perform baseline inference, calculate accuracy, train the model, and evaluate performance using a confusion matrix.

## Tools

Google Colab, Python, CIFAR-10, Ultralytics YOLO, NumPy, PyTorch, and Matplotlib.


In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import random
import os

from google.colab import drive

# Connect Google Drive
drive.mount('/content/drive')

# CIFAR-10 class names
classes = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
]

# Folder where CIFAR-10 will be stored
data_path = "/content/drive/MyDrive/CIFAR-10"

# Convert images to tensors
transform = transforms.ToTensor()

# Load CIFAR-10 from Google Drive
test_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=False,
    download=True,
    transform=transform
)

print("CIFAR-10 dataset loaded successfully.")
print("Number of test images:", len(test_dataset))
print("Number of classes:", len(classes))

In [ ]:
# Display one sample image
image, label = test_dataset[0]

plt.figure(figsize=(3, 3))
plt.imshow(image.permute(1, 2, 0))
plt.title(f"Label: {classes[label]}")
plt.axis("off")
plt.show()

## 2. Random Sampling

Randomly select 10 images from the CIFAR-10 test dataset.


In [ ]:
# Select 10 random images
random_indices = random.sample(range(len(test_dataset)), 10)

sample_images = []
sample_labels = []

for index in random_indices:
    image, label = test_dataset[index]
    sample_images.append(image)
    sample_labels.append(label)

print("Selected indices:", random_indices)
print("Number of selected images:", len(sample_images))

In [ ]:
plt.figure(figsize=(12, 8))

for i in range(10):
    plt.subplot(2, 5, i + 1)

    image = sample_images[i]
    label = sample_labels[i]

    plt.imshow(image.permute(1, 2, 0))
    plt.title(classes[label])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 3. YOLO Classification Inference

Run a pre-trained YOLO classification model on the 10 randomly selected CIFAR-10 images.

In [ ]:
!pip install -q ultralytics

In [ ]:
from ultralytics import YOLO

# Load a pre-trained YOLO classification model
model = YOLO("yolo26n-cls.pt")

In [ ]:
# Convert CIFAR-10 tensors to PIL images
from PIL import Image as PILImage
import numpy as np

pil_images = []

for img in sample_images:
    img_np = (
        img.permute(1, 2, 0).numpy() * 255
    ).astype(np.uint8)

    pil_img = PILImage.fromarray(img_np)
    pil_images.append(pil_img)

print("Images converted successfully.")

In [ ]:
results = model(pil_images)

print("Inference completed.")


In [ ]:
# Display YOLO predictions
for i, result in enumerate(results):
    predicted_class = result.names[result.probs.top1]
    confidence = float(result.probs.top1conf)

    print(
        f"Image {i+1}: "
        f"Prediction = {predicted_class} | "
        f"Confidence = {confidence:.2f}"
    )

## 4. Baseline Accuracy

Compare the YOLO predictions with the CIFAR-10 ground truth labels using compatible class mappings.

In [ ]:
# CIFAR-10 ground truth labels
ground_truth = [classes[label] for label in sample_labels]

# YOLO predictions
predictions = []

for result in results:
    predicted_class = result.names[result.probs.top1]
    predictions.append(predicted_class)

# Display comparison
for i in range(10):
    print(
        f"Image {i+1}: "
        f"Ground Truth = {ground_truth[i]} | "
        f"YOLO = {predictions[i]}"
    )

In [ ]:
# Calculate baseline accuracy
correct = sum(
    ground_truth[i] == predictions[i]
    for i in range(len(ground_truth))
)

accuracy = correct / len(ground_truth)

print(f"Correct predictions: {correct}/{len(ground_truth)}")
print(f"Baseline Accuracy: {accuracy * 100:.2f}%")

The baseline YOLO model was pre-trained on ImageNet classes, which differ from the CIFAR-10 classes. Therefore, the predictions were compared directly with the CIFAR-10 ground-truth labels.

## Part 2: Model Training and Evaluation

Train a YOLO classification model on the CIFAR-10 dataset and evaluate its performance using a multi-class confusion matrix.

## 5. Prepare CIFAR-10 Dataset

Convert the CIFAR-10 dataset into a folder structure compatible with YOLO classification.

In [ ]:
import os

# Main dataset directory
base_dir = "/content/cifar10_yolo"

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

# Create class folders
for class_name in classes:
    os.makedirs(os.path.join(train_dir, class_name), exist_ok=True)
    os.makedirs(os.path.join(val_dir, class_name), exist_ok=True)

print("YOLO dataset folders created.")

## 6. Convert CIFAR-10 to YOLO Format

Convert CIFAR-10 images into PNG files and organize them by class.

In [ ]:
import os
import torchvision

# CIFAR-10 location
data_path = "/content/drive/MyDrive/CIFAR-10"

# YOLO dataset location
base_dir = "/content/drive/MyDrive/cifar10_yolo"

train_dir = os.path.join(base_dir, "train")
val_dir = os.path.join(base_dir, "val")

# Create class folders
for class_name in classes:
    os.makedirs(
        os.path.join(train_dir, class_name),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(val_dir, class_name),
        exist_ok=True
    )

# Load CIFAR-10 from Google Drive
train_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=True,
    download=False
)

val_dataset = torchvision.datasets.CIFAR10(
    root=data_path,
    train=False,
    download=False
)

# Convert training images
for i, (img, label) in enumerate(train_dataset):

    class_name = classes[label]

    save_path = os.path.join(
        train_dir,
        class_name,
        f"image_{i}.png"
    )

    img.save(save_path)

# Convert validation images
for i, (img, label) in enumerate(val_dataset):

    class_name = classes[label]

    save_path = os.path.join(
        val_dir,
        class_name,
        f"image_{i}.png"
    )

    img.save(save_path)

print("CIFAR-10 converted to YOLO format successfully!")
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

## 7. Train YOLO26 Classification Model

Fine-tune a YOLO26 classification model using the CIFAR-10 training dataset.

In [ ]:
from ultralytics import YOLO

# Load pre-trained YOLO26 classification model
model = YOLO("yolo26n-cls.pt")

# Train on CIFAR-10
results = model.train(
    data="/content/drive/MyDrive/cifar10_yolo",
    epochs=5,
    imgsz=32,
    batch=64,
    name="cifar10_yolo_training"
)


In [ ]:
import shutil
import os

source = "/content/runs/classify/cifar10_yolo_training/weights/best.pt"
destination = "/content/drive/MyDrive/best_cifar10_yolo.pt"

shutil.copy(source, destination)

print("Model saved successfully!")
print(destination)

## 8. Confusion Matrix

Evaluate the trained YOLO26 model on the CIFAR-10 validation set and generate a multi-class confusion matrix.

In [ ]:
metrics = best_model.val(
    data="/content/drive/MyDrive/cifar10_yolo",
    plots=True
)

print(f"Top-1 Accuracy: {metrics.top1:.3f}")
print(f"Top-5 Accuracy: {metrics.top5:.3f}")

## 9. Confusion Matrix

Visualize the multi-class confusion matrix for the trained YOLO26 model.

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename="/content/runs/classify/val/confusion_matrix.png""
    )
)

## 10. Confusion Matrix Analysis

The trained YOLO26 model achieved a Top-1 accuracy of 77.3% and a Top-5 accuracy of 98.7% on the CIFAR-10 validation set. The confusion matrix shows the classification performance across all 10 classes and highlights which classes are most frequently confused with each other.